## Model Serving

As the class practice, the students will be required to develop local inference server using the `Churn_Modelling_train_test.csv` dataset and MLFlow for online and batch inference.

**About dataset**

This dataset is obained from [kaggle](https://www.kaggle.com/datasets/shubhammeshram579/bank-customer-churn-prediction?resource=download). It contains information on bank customers who either left the bank or continue to be a customer. The dataset includes the following attributes:

* Customer ID: A unique identifier for each customer
* Surname: The customer's surname or last name
* Credit Score: A numerical value representing the customer's credit score
* Geography: The country where the customer resides (France, Spain or Germany)
* Gender: The customer's gender (Male or Female)
* Age: The customer's age.
* Tenure: The number of years the customer has been with the bank
* Balance: The customer's account balance
* NumOfProducts: The number of bank products the customer uses (e.g., savings account, credit card)
* HasCrCard: Whether the customer has a credit card (1 = yes, 0 = no)
* IsActiveMember: Whether the customer is an active member (1 = yes, 0 = no)
* EstimatedSalary: The estimated salary of the customer
* Exited: Whether the customer has churned (1 = yes, 0 = no)

### Model Training

For this exercice, it is necessary to have a model registered in MLFlow. For this we can, we can use the experiments from session 2.

In [8]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:8081")

In [9]:
import os
import pandas as pd

# Check if the file exists in current directory
print("Current directory:", os.getcwd())
print("\nFiles in current directory:")
for file in os.listdir('.'):
    if 'churn' in file.lower() or 'csv' in file.lower():
        print(f"  - {file}")

Current directory: /Users/mac/Desktop/Eada Masters/Semester 3/ML operations and System Design /Excercises /mlops-and-system-design/class 3 exercise /Class Exercise

Files in current directory:
  - Churn_Modelling_val.csv
  - Churn_Modelling_train_test.csv


In [10]:
# import libraries
import os
import pandas as pd
import numpy as np
import joblib
import requests
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score
import mlflow
from mlflow.models import infer_signature

In [15]:
# Load the dataset
df = pd.read_csv('Churn_Modelling_train_test.csv')  # Adjust filename if different

# Display first few rows to understand the data
print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nColumn names:")
print(df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (9001, 14)

First 5 rows:
   RowNumber  CustomerId        Surname  CreditScore Geography  Gender   Age  \
0       4784    15729224       Jennings          710    France  Female  37.0   
1       1497    15799156     Okwuadigbo          569     Spain    Male  38.0   
2       1958    15674922        Beavers          710    France    Male  54.0   
3       9174    15653572       Thornton          673     Spain    Male  43.0   
4       9748    15775761  Iweobiegbunam          610   Germany  Female  69.0   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       5       0.00              2        1.0             0.0   
1       8       0.00              2        0.0             0.0   
2       6  171137.62              1        1.0             1.0   
3       8  127132.96              1        0.0             1.0   
4       5   86038.21              3        0.0             0.0   

   EstimatedSalary  Exited  
0        115403.31       0  
1         79618.79     

Start the MLflow server with the following command in the terminal: `mlflow server --host 127.0.0.1 --port 8080`.

Now, for the purpose of this exercice, you are required to define again the data transformation logic and save the one hot encoder as a `.pkl` file (if encoder was used during the pipeline).

In [13]:
class Transformer:
    def __init__(self):
        self.DROP_COLUMNS = ['CustomerId', 'Surname', 'RowNumber']
        self.CATEGORICAL_COLS = ['Geography', 'Gender']
        self.BINARY_FEATURES = ['HasCrCard', 'IsActiveMember']
    
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df = df.drop(columns=self.DROP_COLUMNS, errors='ignore')
        
        # Fill NaN values in binary columns with 0 first
        for col in self.BINARY_FEATURES:
            if col in df.columns:
                df[col] = df[col].fillna(0)  # Fill NaN with 0
                df[col] = df[col].astype(int)
        
        # Fill any other NaN values in all columns with 0
        df = df.fillna(0)
        
        df = pd.get_dummies(df, columns=self.CATEGORICAL_COLS, drop_first=False)
        return df

# Apply transformation
transformer = Transformer()
df_transformed = transformer.transform(df)
print("Columns after transformation:", df_transformed.columns.tolist())

Columns after transformation: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_0', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'Gender_Female', 'Gender_Male']


/var/folders/m3/gtxgx72s3qb0bls8dd96lsy40000gn/T/ipykernel_15628/2589498031.py:14: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[col] = df[col].fillna(0)  # Fill NaN with 0
/var/folders/m3/gtxgx72s3qb0bls8dd96lsy40000gn/T/ipykernel_15628/

In [16]:
# Perform another experiment if you don't have the ones from session 2. Otherwise, this part can be skipped

# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

In [17]:
# import validation dataset to test inference
df_validation = pd.read_csv('Churn_Modelling_val.csv')
print(f"Validation shape: {df_validation.shape}")

Validation shape: (1001, 14)


### Inference

In this part, you are asked to implement a function for batch and online inference methods by providing a model uri. 

In [18]:
# import validation dataset to test inference
df_validation = pd.read_csv('Churn_Modelling_val.csv')
print(f"Validation shape: {df_validation.shape}")

Validation shape: (1001, 14)


Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [19]:
# transform data - if necessary
transformer = Transformer()
df_val_transformed = transformer.transform(df_validation)
df_val_transformed = df_val_transformed.fillna(0)
print(f"Transformed shape: {df_val_transformed.shape}")

Transformed shape: (1001, 14)


/var/folders/m3/gtxgx72s3qb0bls8dd96lsy40000gn/T/ipykernel_15628/2589498031.py:14: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df[col] = df[col].fillna(0)  # Fill NaN with 0
/var/folders/m3/gtxgx72s3qb0bls8dd96lsy40000gn/T/ipykernel_15628/

##### Batch Inference

In [26]:
# define a function to implement batch inference with mlflow
def batch_inference(model_uri: str, input_df: pd.DataFrame):
    model = mlflow.pyfunc.load_model(model_uri)
    predictions = model.predict(input_df)
    return predictions

In [34]:
import joblib

# Use the model path from your terminal
model_path = "/Users/mac/Desktop/Eada Masters/Semester 3/ML operations and System Design /Excercises /mlops-and-system-design/mlartifacts/2/models/m-59d5247228d74c1399bcecdd9a2144c6/artifacts/model.pkl"

# Load the model
model = joblib.load(model_path)

# Get expected columns
expected_columns = model.feature_names_in_

# Align validation data
df_val_transformed_aligned = df_val_transformed.reindex(columns=expected_columns, fill_value=0)

# Make predictions
batch_prediction_result = model.predict(df_val_transformed_aligned)
print(f"First 10 predictions: {batch_prediction_result[:10]}")
print(f"Predictions shape: {batch_prediction_result.shape}")


First 10 predictions: [0 0 1 0 0 1 0 0 0 1]
Predictions shape: (1001,)


In [35]:
# check the confusion matrix
from sklearn.metrics import confusion_matrix
y_true = df_validation['Exited']
cm = confusion_matrix(y_true, batch_prediction_result)
print("Confusion Matrix:")
print(cm)
print(f"\nTrue Negatives: {cm[0,0]}, False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}, True Positives: {cm[1,1]}")

Confusion Matrix:
[[628 175]
 [ 50 148]]

True Negatives: 628, False Positives: 175
False Negatives: 50, True Positives: 148


##### Online Inference

For the online inference, it is required to set up local server. Follow the steps below to configure it:

1. Open a new bash terminal
2. Execute the follwing command `export MLFLOW_TRACKING_URI=http://127.0.0.1:8080` in the terminal. You should specify the port that we are using for MLFlow
3. Execute the following command `mlflow models serve -m runs:/<run_id>/model -p 5000 --no-conda`. Note that `runs:/<run_id>/model` is your model uri.

In [ ]:
import requests
import json

In [ ]:
# import validation dataset to test inference - just one record
single_record = df_val_transformed.head(1)
print(f"Single record shape: {single_record.shape}")

Note that the data might need to be transformed to match the model schema. You can check the schema in the `input_example.json` file in MLFlow. 

In [ ]:
def get_inference_endpoint(host="http://127.0.0.1", port=5000):
    return f"{host}:{port}/invocations"

url = get_inference_endpoint()

In [ ]:
# define a function to implement online inference with mlflow - pandas input
def online_inference_pandas(url: str, input_df: pd.DataFrame):
    data_json = input_df.to_json(orient='split')
    response = requests.post(url, headers={'Content-Type': 'application/json'}, data=data_json)
    return response

In [ ]:
url = "http://127.0.0.1:5000/invocations"
response_pandas = online_inference_pandas(url, single_record)
print("Response status:", response_pandas.status_code)
print("Prediction:", response_pandas.json())

In [ ]:
# define a function to implement online inference with mlflow - json input
def online_inference_json(url: str, input_dict: dict):
    response = requests.post(url, headers={'Content-Type': 'application/json'}, json=input_dict)
    return response

In [ ]:
# define the json as required by MLFlow
input_json = {
    "dataframe_split": {
        "columns": single_record.columns.tolist(),
        "data": single_record.values.tolist()
    }
}

response_json = online_inference_json(url, input_json)
print("Response status:", response_json.status_code)
print("Prediction:", response_json.json())
}

In [ ]:
response_json = online_inference_json()
response_json.content